In [ ]:
# ============================================================
# DAIS-C: AUTOMATED DISORGANIZED-SPEECH QUANTIFICATION
# Raw text -> FTD-related annotation probability
# Participant-level cross-validation
# ============================================================

!pip -q install sentence-transformers scikit-learn striprtf python-docx

import os, re, glob
import numpy as np
import pandas as pd
from docx import Document
from striprtf.striprtf import rtf_to_text

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ------------------------------------------------------------
# 1. DAIS 원본에서 annotation 다시 읽기
# ------------------------------------------------------------

ROOT = "/content/DAIS_EXTRACTED"

PID_RE = re.compile(r"(\d{2}[A-Za-z]{2}\d{2})")
TAG_RE = re.compile(r"</?[^>]+>")

def pid_from_path(p):
    m = PID_RE.search(os.path.basename(p))
    return m.group(1).upper() if m else None

def read_file(path):
    if path.lower().endswith(".docx"):
        return [p.text.strip() for p in Document(path).paragraphs
                if p.text.strip()]

    if path.lower().endswith(".rtf"):
        with open(path, encoding="utf-8", errors="ignore") as f:
            return [x.strip() for x in
                    rtf_to_text(f.read()).splitlines() if x.strip()]

    return []

def clean(x):
    x = TAG_RE.sub("", x)
    return re.sub(r"\s+", " ", x).strip()

# annotation 포함 speaker/full 파일 찾기
files = []

for p in glob.glob(ROOT + "/**/*", recursive=True):

    if not os.path.isfile(p):
        continue

    if not p.lower().endswith((".docx",".rtf")):
        continue

    s = p.lower().replace("\\","/")

    # annotation이 있는 파일
    if (
        "speaker only for analysis" in s
        or "speaker_only_for_analysis" in s
        or "full" in os.path.basename(s)
    ):
        files.append(p)

# participant별 하나만 선택
chosen = {}

for p in files:
    pid = pid_from_path(p)
    if pid and pid not in chosen:
        chosen[pid] = p

print("Annotated participants found:", len(chosen))

# ------------------------------------------------------------
# 2. Annotation dataset 생성
# ------------------------------------------------------------

rows = []

for pid, path in chosen.items():

    group = "CL" if "dai-c-cl" in path.lower() else "CO"

    for line in read_file(path):

        # interviewer 제외
        if "<INT>" in line.upper():
            continue

        txt = clean(line)

        if not txt:
            continue

        gr = int(bool(re.search(r"<Gr(?:\s|>)", line, re.I)))
        ws = int(bool(re.search(r"<WS(?:\s|>)", line, re.I)))
        tc = int(bool(re.search(r"<TC(?:\s|>)", line, re.I)))
        dt = int(bool(re.search(r"<DT(?:\s|>)", line, re.I)))

        rows.append({
            "participant_id": pid,
            "group": group,
            "text": txt,
            "Gr": gr,
            "WS": ws,
            "TC": tc,
            "DT": dt,
            "FTD_event": int(gr or ws or tc or dt)
        })

df = pd.DataFrame(rows)

# 너무 짧은 backchannel 제거
df["word_count"] = df["text"].str.findall(
    r"\b[\w'-]+\b"
).str.len()

df = df[df.word_count >= 3].reset_index(drop=True)

print("\nParticipants:", df.participant_id.nunique())
print("Utterances:", len(df))

print("\nAnnotation prevalence:")
print(
    df[["Gr","WS","TC","DT","FTD_event"]]
    .mean()
    .round(3)
)

print("\nEvents:")
print(df.FTD_event.value_counts())


# ------------------------------------------------------------
# 3. 이미 계산한 SBERT 대신 여기서 정확히 다시 embedding
# annotation dataframe와 순서 맞춤
# ------------------------------------------------------------

from sentence_transformers import SentenceTransformer
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device=device
)

X = model.encode(
    df.text.tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

y = df.FTD_event.values
groups = df.participant_id.values


# ------------------------------------------------------------
# 4. Leave-One-Participant-Out prediction
# ------------------------------------------------------------

logo = LeaveOneGroupOut()

pred = np.full(len(df), np.nan)

for train_idx, test_idx in logo.split(X, y, groups):

    # training set에 두 class가 있어야 함
    if len(np.unique(y[train_idx])) < 2:
        continue

    clf = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=0.1,
            class_weight="balanced",
            max_iter=3000
        )
    )

    clf.fit(X[train_idx], y[train_idx])

    pred[test_idx] = clf.predict_proba(
        X[test_idx]
    )[:,1]


df["disorganization_probability"] = pred

valid = df.disorganization_probability.notna()

# ------------------------------------------------------------
# 5. 전체 out-of-sample 성능
# ------------------------------------------------------------

auc = roc_auc_score(
    df.loc[valid,"FTD_event"],
    df.loc[valid,"disorganization_probability"]
)

ap = average_precision_score(
    df.loc[valid,"FTD_event"],
    df.loc[valid,"disorganization_probability"]
)

baseline = df.loc[valid,"FTD_event"].mean()

print("\n" + "="*60)
print("OUT-OF-PARTICIPANT VALIDATION")
print("="*60)

print(f"ROC-AUC : {auc:.3f}")
print(f"PR-AUC  : {ap:.3f}")
print(f"Event prevalence baseline: {baseline:.3f}")


# ------------------------------------------------------------
# 6. 실제 annotated vs non-annotated probability
# ------------------------------------------------------------

print("\nMean predicted probability:")

print(
    df.groupby("FTD_event")
      ["disorganization_probability"]
      .mean()
      .round(3)
)


# ------------------------------------------------------------
# 7. Participant-level Continuous Disorganization Burden
# ------------------------------------------------------------

P = (
    df.groupby(["participant_id","group"])
      .agg(
          n_utterances=("text","size"),
          annotated_event_rate=("FTD_event","mean"),
          disorganization_burden=(
              "disorganization_probability","mean"
          )
      )
      .reset_index()
)

print("\n" + "="*60)
print("CONTINUOUS DISORGANIZATION BURDEN")
print("="*60)

print(
    P.groupby("group")[
        ["annotated_event_rate",
         "disorganization_burden"]
    ].agg(["mean","std"]).round(3)
)


# ------------------------------------------------------------
# 8. Annotation type별 probability
# ------------------------------------------------------------

print("\n" + "="*60)
print("PROBABILITY BY DAIS-C ANNOTATION")
print("="*60)

for tag in ["Gr","WS","TC","DT"]:

    a = df[df[tag]==1]["disorganization_probability"].mean()
    b = df[df[tag]==0]["disorganization_probability"].mean()

    print(
        f"{tag}: annotated={a:.3f} | "
        f"not annotated={b:.3f} | "
        f"Δ={a-b:+.3f}"
    )


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

df.to_csv(
    "/content/DAIS_C_DISORGANIZATION_UTTERANCES.csv",
    index=False
)

P.to_csv(
    "/content/DAIS_C_DISORGANIZATION_PARTICIPANTS.csv",
    index=False
)

print("\n✓ DONE")
print("Saved:")
print("/content/DAIS_C_DISORGANIZATION_UTTERANCES.csv")
print("/content/DAIS_C_DISORGANIZATION_PARTICIPANTS.csv")

In [ ]:
# ============================================================
# FINAL DIMENSION-SPECIFIC VALIDATION
# Gr / WS / TC / DT 각각 Leave-One-Participant-Out
# 이전 셀의 df, X, groups 그대로 사용
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

logo = LeaveOneGroupOut()
tags = ["Gr", "WS", "TC", "DT"]

results = []
all_predictions = df[
    ["participant_id", "group", "text"]
].copy()

for tag in tags:

    y = df[tag].values
    pred = np.full(len(df), np.nan)

    print("\n" + "="*60)
    print(f"{tag} ANALYSIS")
    print("="*60)

    print(
        f"Positive events: {y.sum()} / {len(y)} "
        f"({y.mean():.3f})"
    )

    # --------------------------------------------------------
    # Leave-One-Participant-Out
    # --------------------------------------------------------
    for train_idx, test_idx in logo.split(X, y, groups):

        # training data에 두 class 모두 있어야 함
        if len(np.unique(y[train_idx])) < 2:
            continue

        clf = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=0.1,
                class_weight="balanced",
                max_iter=3000
            )
        )

        clf.fit(X[train_idx], y[train_idx])

        pred[test_idx] = clf.predict_proba(
            X[test_idx]
        )[:, 1]

    valid = ~np.isnan(pred)

    auc = roc_auc_score(
        y[valid],
        pred[valid]
    )

    ap = average_precision_score(
        y[valid],
        pred[valid]
    )

    prevalence = y[valid].mean()

    pos_mean = pred[valid & (y == 1)].mean()
    neg_mean = pred[valid & (y == 0)].mean()

    lift = ap / prevalence

    results.append({
        "dimension": tag,
        "positive_n": int(y.sum()),
        "prevalence": prevalence,
        "ROC_AUC": auc,
        "PR_AUC": ap,
        "PR_lift_vs_baseline": lift,
        "pred_positive_mean": pos_mean,
        "pred_negative_mean": neg_mean,
        "difference": pos_mean - neg_mean
    })

    all_predictions[f"{tag}_probability"] = pred

    print(f"ROC-AUC : {auc:.3f}")
    print(
        f"PR-AUC  : {ap:.3f} "
        f"(baseline={prevalence:.3f})"
    )
    print(f"PR lift : {lift:.2f}x")
    print(
        f"Predicted probability: "
        f"annotated={pos_mean:.3f} | "
        f"non-annotated={neg_mean:.3f}"
    )


# ============================================================
# SUMMARY
# ============================================================

R = pd.DataFrame(results)

print("\n\n" + "="*72)
print("FINAL DIMENSION-SPECIFIC RESULTS")
print("="*72)

print(
    R.round(3).to_string(index=False)
)


# ============================================================
# Participant-level dimension scores
# ============================================================

prob_cols = [
    "Gr_probability",
    "WS_probability",
    "TC_probability",
    "DT_probability"
]

P = (
    all_predictions
    .groupby(["participant_id", "group"])[prob_cols]
    .mean()
    .reset_index()
)

print("\n" + "="*72)
print("PARTICIPANT-LEVEL CONTINUOUS SCORES")
print("="*72)

print(
    P.groupby("group")[prob_cols]
    .mean()
    .round(3)
)


# ============================================================
# SAVE
# ============================================================

R.to_csv(
    "/content/DAIS_C_DIMENSION_RESULTS.csv",
    index=False
)

all_predictions.to_csv(
    "/content/DAIS_C_DIMENSION_PREDICTIONS.csv",
    index=False
)

P.to_csv(
    "/content/DAIS_C_DIMENSION_PARTICIPANTS.csv",
    index=False
)

print("\n✓ DONE")

In [ ]:
# ============================================================
# FINAL STATISTICAL VALIDATION
# Participant-bootstrap 95% CI + participant-level permutation
# 기존 df / predictions 재사용 → 모델 다시 학습 안 함
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

np.random.seed(42)

# ------------------------------------------------------------
# 1. 분석할 5개 outcome
# ------------------------------------------------------------

tests = {
    "Any_FTD": ("FTD_event", "disorganization_probability"),
    "Gr": ("Gr", "Gr_probability"),
    "WS": ("WS", "WS_probability"),
    "TC": ("TC", "TC_probability"),
    "DT": ("DT", "DT_probability")
}

# 앞 셀의 dimension prediction 붙이기
D = df.copy()

for col in [
    "Gr_probability",
    "WS_probability",
    "TC_probability",
    "DT_probability"
]:
    D[col] = all_predictions[col].values


# ------------------------------------------------------------
# 2. Participant bootstrap CI
# participant 자체를 sampling unit으로 사용
# ------------------------------------------------------------

def participant_bootstrap_auc(data, ycol, pcol, B=5000):

    participants = data["participant_id"].unique()
    aucs = []

    for _ in range(B):

        sampled = np.random.choice(
            participants,
            size=len(participants),
            replace=True
        )

        pieces = []

        # 같은 participant가 여러 번 뽑혀도
        # 독립 bootstrap cluster로 취급
        for i, pid in enumerate(sampled):
            x = data[data.participant_id == pid].copy()
            x["_boot_id"] = i
            pieces.append(x)

        boot = pd.concat(pieces, ignore_index=True)

        y = boot[ycol].values
        p = boot[pcol].values

        valid = ~pd.isna(p)

        y = y[valid]
        p = p[valid]

        if len(np.unique(y)) < 2:
            continue

        aucs.append(
            roc_auc_score(y, p)
        )

    return (
        np.percentile(aucs, 2.5),
        np.percentile(aucs, 97.5)
    )


# ------------------------------------------------------------
# 3. Participant-level permutation
#
# IMPORTANT:
# annotation labels 전체를 participant 단위로 이동시킴.
# 한 사람 내부의 annotation structure는 보존.
# ------------------------------------------------------------

def participant_permutation_auc(
    data,
    ycol,
    pcol,
    B=5000
):

    participants = data["participant_id"].unique()

    observed = roc_auc_score(
        data[ycol],
        data[pcol]
    )

    perm_aucs = []

    # participant별 label sequence 저장
    label_blocks = {
        pid: data.loc[
            data.participant_id == pid,
            ycol
        ].values
        for pid in participants
    }

    # participant마다 발화 수가 달라서
    # 직접 block swap은 불가능.
    #
    # 따라서 각 participant 내부에서 prediction을
    # label과 독립적으로 circular shift.
    # 개인별 prevalence와 temporal structure를 최대한 보존.

    for _ in range(B):

        y_perm = np.empty(len(data), dtype=float)

        for pid in participants:

            idx = np.where(
                data.participant_id.values == pid
            )[0]

            y = data.iloc[idx][ycol].values

            if len(y) <= 1:
                y_perm[idx] = y
                continue

            shift = np.random.randint(0, len(y))

            y_perm[idx] = np.roll(y, shift)

        valid = ~pd.isna(data[pcol].values)

        yp = y_perm[valid]
        pp = data[pcol].values[valid]

        if len(np.unique(yp)) < 2:
            continue

        perm_aucs.append(
            roc_auc_score(yp, pp)
        )

    perm_aucs = np.array(perm_aucs)

    # one-sided: AUC가 우연보다 큰가?
    pvalue = (
        np.sum(perm_aucs >= observed) + 1
    ) / (len(perm_aucs) + 1)

    return observed, pvalue, perm_aucs.mean()


# ------------------------------------------------------------
# 4. 실행
# ------------------------------------------------------------

results = []

for name, (ycol, pcol) in tests.items():

    tmp = D[
        ["participant_id", ycol, pcol]
    ].dropna().reset_index(drop=True)

    auc = roc_auc_score(
        tmp[ycol],
        tmp[pcol]
    )

    lo, hi = participant_bootstrap_auc(
        tmp, ycol, pcol
    )

    obs, p, null_auc = participant_permutation_auc(
        tmp, ycol, pcol
    )

    results.append({
        "dimension": name,
        "AUC": auc,
        "CI_low": lo,
        "CI_high": hi,
        "permutation_p": p,
        "null_mean_AUC": null_auc
    })


R_final = pd.DataFrame(results)

# BH-FDR
pvals = R_final["permutation_p"].values
order = np.argsort(pvals)

qvals = np.empty(len(pvals))
ranked = pvals[order]

adjusted = ranked * len(pvals) / np.arange(1, len(pvals)+1)
adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
adjusted = np.minimum(adjusted, 1)

qvals[order] = adjusted

R_final["FDR_q"] = qvals


print("\n" + "="*78)
print("FINAL STATISTICAL VALIDATION")
print("="*78)

print(
    R_final.round(4).to_string(index=False)
)

R_final.to_csv(
    "/content/DAIS_C_FINAL_VALIDATION.csv",
    index=False
)

print("\n✓ Saved: /content/DAIS_C_FINAL_VALIDATION.csv")